# Day 20 Capstone Mini-Project: CIFAR-10 Image Classifier with Transfer Learning

This notebook completes the full assignment in simple steps:
1. Load a small, balanced CIFAR-10 subset and visualize it.
2. Train a small CNN from scratch as the baseline.
3. Create data augmentation.
4. Use MobileNetV2 for transfer learning (feature extraction).
5. Fine-tune the top layers of MobileNetV2.
6. Evaluate using accuracy, confusion matrix, precision, recall and F1-score.
7. Interpret predictions using Grad-CAM.
8. Print a short comparison/reflection automatically from the model results.

**Important:** The assignment environment says CIFAR-10 and MobileNetV2 weights are already cached locally. Run the notebook cells from top to bottom.


## Prerequisite: imports, reproducibility, and balanced CIFAR-10 subset
We keep only **500 training images per class** and **100 test images per class** so training remains practical on CPU.


In [ ]:
# ================================
# IMPORT LIBRARIES
# ================================
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Use the same random seed each time so results are more reproducible.
RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)

# Load CIFAR-10 from the local Keras cache used by the lab.
# CIFAR-10 contains 32x32 colour images from 10 classes.
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()

# Human-readable names for label numbers 0 to 9.
CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Create a balanced subset: the same number of images from every class.
def subset(x, y, per_class):
    y = y.ravel()  # Change labels from shape (n,1) to (n,)

    # Pick 'per_class' random indexes for every class without duplicates.
    idx = np.hstack([
        np.random.RandomState(RNG).choice(
            np.where(y == c)[0], per_class, replace=False
        )
        for c in range(10)
    ])

    # Shuffle the selected indexes so class images are mixed.
    np.random.RandomState(RNG).shuffle(idx)

    # Scale image pixels from 0-255 to 0-1 for the baseline CNN.
    return x[idx].astype("float32") / 255.0, y[idx].astype("int32")

# 10 classes x 500 = 5000 training images.
x_train, y_train = subset(x_train_full, y_train_full, 500)

# 10 classes x 100 = 1000 test images.
x_test, y_test = subset(x_test_full, y_test_full, 100)

print("Training shape:", x_train.shape)
print("Test shape    :", x_test.shape)
print("Pixel range   :", x_train.min(), "to", x_train.max())


## Step 1: Load and visualize the data
We display 18 labeled examples and verify that every training class contains exactly 500 images.


In [ ]:
# Show 18 sample images from the balanced training subset.
fig, axes = plt.subplots(3, 6, figsize=(9, 4.7))

for ax, i in zip(axes.ravel(), range(18)):
    ax.imshow(x_train[i])
    ax.set_title(CLASSES[y_train[i]], fontsize=8)
    ax.axis("off")

plt.suptitle("CIFAR-10 subset - sample images")
plt.tight_layout()
plt.show()

# Count how many training images belong to each class.
counts = np.bincount(y_train, minlength=10)
print("Training images per class:")
for class_index, count in enumerate(counts):
    print(f"{CLASSES[class_index]:12s}: {count}")


## Step 2: Baseline CNN from scratch
This small CNN learns only from our 5,000 training images. `EarlyStopping` ends training when validation accuracy stops improving.


In [ ]:
# Function that builds a small CNN from scratch.
def make_baseline_cnn():
    model = keras.Sequential([
        # Input image is 32x32 with 3 RGB colour channels.
        keras.Input((32, 32, 3)),

        # Block 1: learn simple features such as edges.
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 2: learn more detailed image patterns.
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 3: learn higher-level features.
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),

        # Convert each feature map into one value.
        layers.GlobalAveragePooling2D(),

        # Dropout reduces overfitting.
        layers.Dropout(0.3),

        # 10 output probabilities, one for each CIFAR-10 class.
        layers.Dense(10, activation="softmax")
    ])

    # Sparse categorical crossentropy is used because y contains integer labels 0-9.
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

baseline = make_baseline_cnn()
baseline.summary()

# Stop if validation accuracy fails to improve for 4 epochs.
es = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=4,
    restore_best_weights=True
)

# Train on 80% of x_train; Keras automatically uses 20% for validation.
hist_base = baseline.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=64,
    callbacks=[es],
    verbose=2
)

# Evaluate the baseline on data that was not used for training.
base_test = baseline.evaluate(x_test, y_test, verbose=0)[1]
print(f"BASELINE test accuracy = {base_test:.3f}")


In [ ]:
# Plot baseline training curves so we can check learning and overfitting.
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(hist_base.history["accuracy"], label="Train")
plt.plot(hist_base.history["val_accuracy"], label="Validation")
plt.title("Baseline accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist_base.history["loss"], label="Train")
plt.plot(hist_base.history["val_loss"], label="Validation")
plt.title("Baseline loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


## Step 3: Data augmentation
Augmentation creates slightly changed versions of training images using horizontal flips, small rotations and zooms. It is active only while training.


In [ ]:
# Data augmentation makes the model see more image variations.
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.1),
], name="augment")

# Take one original image.
img = x_train[1][None, ...]

# Display the original plus four augmented versions.
fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))
axes[0].imshow(img[0])
axes[0].set_title("original", fontsize=9)
axes[0].axis("off")

for ax in axes[1:]:
    augmented = data_augmentation(img, training=True)[0]
    ax.imshow(np.clip(augmented, 0, 1))
    ax.set_title("augmented", fontsize=9)
    ax.axis("off")

plt.suptitle(f"Augmentation preview - {CLASSES[y_train[1]]}")
plt.tight_layout()
plt.show()


## Step 4: Transfer learning, Part A - feature extraction
We use **MobileNetV2 pretrained on ImageNet**. First its pretrained backbone is frozen, so only the new 10-class classifier head learns.

The assignment resizes CIFAR-10 images from 32x32 to 96x96 before passing them to MobileNetV2.


In [ ]:
# MobileNetV2 will receive 96x96 images.
IMG = 96

# Load MobileNetV2 without its original ImageNet classifier.
base = keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze all pretrained MobileNetV2 layers for feature extraction.
base.trainable = False

# Build the full transfer-learning model using the Functional API.
inputs = keras.Input((32, 32, 3), name="cifar_input")

# Apply augmentation during training.
x = data_augmentation(inputs)

# Resize CIFAR-10 from 32x32 to 96x96.
x = layers.Resizing(IMG, IMG)(x)

# Our images are currently 0-1. MobileNetV2 preprocessing expects 0-255 input,
# so first convert back to 0-255 and then call preprocess_input.
x = layers.Rescaling(255.0)(x)
x = keras.applications.mobilenet_v2.preprocess_input(x)

# The frozen MobileNetV2 backbone extracts useful visual features.
# training=False keeps BatchNormalization layers in inference mode.
feature_maps = base(x, training=False)

# Turn the feature maps into one feature vector.
x = layers.GlobalAveragePooling2D()(feature_maps)
x = layers.Dropout(0.3)(x)

# New task-specific output head for the 10 CIFAR-10 classes.
outputs = layers.Dense(10, activation="softmax", name="predictions")(x)

tl_model = keras.Model(inputs, outputs, name="cifar10_mobilenetv2")

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Count the currently trainable parameters.
trainable_params = sum(np.prod(w.shape) for w in tl_model.trainable_weights)
print(f"Trainable parameters (head only): {trainable_params:,}")
tl_model.summary()


In [ ]:
# Save the best model whenever validation accuracy improves.
ckpt = keras.callbacks.ModelCheckpoint(
    "day20_best.keras",
    monitor="val_accuracy",
    save_best_only=True
)

# Stop when validation accuracy no longer improves.
es2 = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

# Train only the new classification head.
hist_fe = tl_model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=8,
    batch_size=64,
    callbacks=[es2, ckpt],
    verbose=2
)

# Test feature-extraction model.
fe_test = tl_model.evaluate(x_test, y_test, verbose=0)[1]
print(f"FEATURE-EXTRACTION test accuracy = {fe_test:.3f}")


## Step 5: Transfer learning, Part B - fine-tuning
Now we unfreeze only the last 30 MobileNetV2 layers. We use a very small learning rate (`1e-5`) so pretrained features are adjusted gently rather than destroyed.


In [ ]:
# Allow MobileNetV2 to contain trainable layers.
base.trainable = True

# Freeze every layer except the final 30 layers.
for layer in base.layers[:-30]:
    layer.trainable = False

# IMPORTANT: compile again after changing layer.trainable values.
# Use a much smaller learning rate for fine-tuning.
tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

es3 = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

# Continue training with the top part of the pretrained backbone unfrozen.
hist_ft = tl_model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=6,
    batch_size=64,
    callbacks=[es3, ckpt],
    verbose=2
)

# Final fine-tuned test accuracy.
ft_test = tl_model.evaluate(x_test, y_test, verbose=0)[1]
print(f"FINE-TUNED test accuracy = {ft_test:.3f}")
print(f"baseline={base_test:.3f}  feature-extract={fe_test:.3f}  fine-tuned={ft_test:.3f}")


## Step 6: Confusion matrix and per-class scores
Accuracy gives one overall number. The confusion matrix and classification report show **which classes are easy or difficult**.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Predict probabilities for all test images.
y_prob = tl_model.predict(x_test, verbose=0)

# argmax selects the class with the highest predicted probability.
y_pred = y_prob.argmax(axis=1)

# Build and display the confusion matrix.
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False
)
plt.title("Confusion matrix - fine-tuned model")
plt.tight_layout()
plt.show()

# Precision: when the model predicts a class, how often is it correct?
# Recall: of all real examples of a class, how many did the model find?
# F1-score: balance between precision and recall.
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3))


## Step 7: Interpret predictions with Grad-CAM
Grad-CAM creates a heatmap showing **where the model looked** when making a prediction. Hot colours indicate regions that influenced the prediction more strongly.


In [ ]:
# We need the output of MobileNetV2's final convolutional feature map.
# Find the transfer model layer that contains the MobileNetV2 backbone.
backbone_layer = next(layer for layer in tl_model.layers if layer.name == base.name)

# Create a model that reproduces preprocessing up to the backbone input.
# This avoids graph-disconnection problems with nested Keras models.
pre_backbone_model = keras.Model(tl_model.input, backbone_layer.input)

# Create a model that maps MobileNetV2 input to its final feature map.
backbone_feature_model = keras.Model(backbone_layer.input, backbone_layer.output)

# Get the layers after MobileNetV2 so we can reproduce the classifier head.
gap_layer = next(layer for layer in tl_model.layers if isinstance(layer, layers.GlobalAveragePooling2D))
dropout_layer = next(layer for layer in tl_model.layers if isinstance(layer, layers.Dropout) and layer.name != "augment")
prediction_layer = tl_model.get_layer("predictions")

# Grad-CAM function for one image.
def gradcam(img_1, pred_index=None):
    # Add a batch dimension: (32,32,3) -> (1,32,32,3)
    img_batch = img_1[None, ...]

    # Produce the exact tensor that enters MobileNetV2.
    backbone_input = pre_backbone_model(img_batch, training=False)

    # Record operations so TensorFlow can calculate gradients.
    with tf.GradientTape() as tape:
        conv_out = backbone_feature_model(backbone_input, training=False)
        tape.watch(conv_out)

        # Reproduce our classification head.
        x_head = gap_layer(conv_out)
        x_head = dropout_layer(x_head, training=False)
        preds = prediction_layer(x_head)

        # If class was not supplied, explain the model's predicted class.
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))

        class_score = preds[:, pred_index]

    # How much did every feature-map position affect the class score?
    grads = tape.gradient(class_score, conv_out)

    # Average the gradients over height and width to get channel weights.
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weighted combination of feature maps gives the Grad-CAM heatmap.
    cam = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    cam = tf.maximum(cam, 0)

    # Normalize safely to the range 0-1.
    max_value = tf.reduce_max(cam)
    cam = cam / (max_value + 1e-8)

    return cam.numpy(), pred_index

# Select three correctly classified test images.
correct = np.where(y_pred == y_test)[0]
show = correct[:3]

if len(show) == 0:
    print("No correct prediction was found to display.")
else:
    fig, axes = plt.subplots(len(show), 3, figsize=(9, 3 * len(show)))

    # Keep axes 2D even if only one image is shown.
    axes = np.atleast_2d(axes)

    for row, i in enumerate(show):
        cam, pidx = gradcam(x_test[i])

        # Resize heatmap to the original 32x32 image size.
        cam_up = tf.image.resize(cam[..., None], (32, 32)).numpy().squeeze()

        axes[row, 0].imshow(x_test[i])
        axes[row, 0].set_title(f"Actual: {CLASSES[y_test[i]]}")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(x_test[i])
        axes[row, 1].imshow(cam_up, cmap="jet", alpha=0.45)
        axes[row, 1].set_title(f"Pred: {CLASSES[pidx]}")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(cam_up, cmap="jet")
        axes[row, 2].set_title("Grad-CAM")
        axes[row, 2].axis("off")

    plt.suptitle("Where the fine-tuned model looks")
    plt.tight_layout()
    plt.show()


## Step 8: Written analysis - baseline vs transfer learning
This final cell prints the required 3-row result summary and creates a short, honest reflection using the actual results from your run.


In [ ]:
# Print the three important accuracy values.
print("=== RESULTS SUMMARY ===")
print(f"{'approach':22s}{'test acc':>10s}")

for name, acc in [
    ("Baseline CNN (scratch)", base_test),
    ("Transfer: feat-extract", fe_test),
    ("Transfer: fine-tuned", ft_test),
]:
    print(f"{name:22s}{acc:10.3f}")

lift = ft_test - base_test
print(f"\nAbsolute lift (fine-tuned - baseline) = {lift:+.3f}")

# Find each class's recall from the confusion matrix.
# Recall = correctly predicted examples / all real examples in that class.
per_class_recall = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)

# Lower recall means that class was harder for the model.
hardest_ids = np.argsort(per_class_recall)[:2]
hardest_text = " and ".join(CLASSES[i] for i in hardest_ids)

# Create an automatic 5-sentence reflection based on the actual run.
if lift > 0:
    comparison_sentence = (
        f"Transfer learning helped: the fine-tuned model achieved {ft_test:.3f} test accuracy "
        f"compared with {base_test:.3f} for the scratch CNN, an absolute improvement of {lift:.3f}."
    )
else:
    comparison_sentence = (
        f"In this run, fine-tuning achieved {ft_test:.3f} test accuracy versus {base_test:.3f} "
        f"for the scratch CNN, so it did not improve the baseline ({lift:+.3f} difference)."
    )

reflection = (
    comparison_sentence + " "
    f"Feature extraction reached {fe_test:.3f}, showing how much the frozen ImageNet features contributed "
    "before the backbone was fine-tuned. "
    f"The hardest classes by test recall were {hardest_text}, which is visible in the confusion matrix through their larger off-diagonal errors. "
    "The Grad-CAM plots show the image regions used for the displayed predictions; heat concentrated on the object is more reassuring than heat concentrated mainly on background. "
    "To improve the model next, I would try more training data and carefully tune the number of unfrozen layers and the fine-tuning learning rate."
)

print("\n=== SHORT REFLECTION ===")
print(reflection)

# Save the final fine-tuned model as requested by the assignment.
tl_model.save("day20_best.keras")
print("\nSaved final model -> day20_best.keras")


## Final checklist
- Balanced 5,000-image training subset and 1,000-image test subset
- Labeled data visualization and class counts
- CNN baseline with EarlyStopping
- Data augmentation preview
- MobileNetV2 feature extraction
- Fine-tuning of the last 30 backbone layers with low learning rate
- Accuracy, confusion matrix, precision, recall and F1-score
- Grad-CAM interpretation
- Automatic results summary and 5-sentence reflection
- Final model saved as `day20_best.keras`

**Submission:** Run all cells in order so every output/plot is visible, quickly read the generated Step 8 reflection, then save the executed notebook before submitting.
